# Manna Trade Analysis

This notebook loads **real trade data** from the Manna API (simulation or live) and runs simple analytics.

**Prerequisites:** Start the Manna app (`npm run dev`) so `http://localhost:3000/api/export` is available, or set `BASE_URL` to your deployed URL.

In [ ]:
import requests
import pandas as pd

BASE_URL = "http://localhost:3000"  # or https://your-domain.com
r = requests.get(f"{BASE_URL}/api/export?format=json&limit=500&days=30", timeout=10)
r.raise_for_status()
data = r.json()
if not data.get("success") or "data" not in data:
    raise SystemExit("API returned no data. Is the app running?")
trades = data["data"]["trades"]
df = pd.DataFrame(trades)
print(f"Loaded {len(df)} trades")
df.head()

## Summary stats

In [ ]:
if len(df) == 0:
    print("No trades yet. Run the agent runner to generate simulated trades.")
else:
    df["pnl"] = pd.to_numeric(df["pnl"], errors="coerce").fillna(0)
    total_pnl = df["pnl"].sum()
    wins = (df["pnl"] > 0).sum()
    losses = (df["pnl"] < 0).sum()
    win_rate = (wins / len(df)) * 100 if len(df) else 0
    print(f"Total P&L: {total_pnl:.2f}")
    print(f"Win rate: {win_rate:.1f}% ({wins}W / {losses}L)")
    print(f"Simulation stats (if available):", data["data"].get("stats"))

## P&L by symbol

In [ ]:
if len(df) > 0:
    by_symbol = df.groupby("symbol")["pnl"].agg(["sum", "count"]).round(2)
    by_symbol.columns = ["total_pnl", "trades"]
    by_symbol.sort_values("total_pnl", ascending=False)